In [ ]:
using CSV, DataFrames, Statistics, Printf, StatsBase
include("functions.jl")

data_dir  = "output"
burnin    = 500_000
thin      = 10
chain_ids = 1:3
outfile   = joinpath(data_dir, "rhat_summary.csv")
waic_path = joinpath(data_dir, "waic_best3_mergedchains.csv")
@assert isfile(waic_path) "File not found: $(waic_path)"
waic_df = CSV.read(waic_path, DataFrame)

@assert all(x -> x in names(waic_df), ["Dataset","best_k"]) "waic_best3_mergedchains.csv must contain columns: Dataset,best_k"

function parse_best_k(x)
    if x isa Missing
        return nothing
    elseif x isa AbstractString
        try
            return parse(Int, x)
        catch
            return nothing
        end
    elseif x isa Number
        if isnan(x)
            return nothing
        else
            return Int(round(x))
        end
    else
        return nothing
    end
end

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    if nrow(df) <= burnin
        @warn "File has only $(nrow(df)) rows (<= burn-in): $path"
        return DataFrame()  
    end
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

function rhat_for_dataset(sim::Int, best_k::Int, chain_ids; burnin::Int, thin::Int)
    dfs = DataFrame[]
    present_chains = Int[]
    for c in chain_ids
        fpath = joinpath(data_dir, @sprintf("samples_sim_%d_chain_%d_k%d.csv", sim, c, best_k))
        if !isfile(fpath)
            @warn "Missing samples file: $fpath"
            continue
        end
        df = load_chain_df(fpath; burnin=burnin, thin=thin)
        if nrow(df) == 0
            @warn "No usable rows after burn/thin: $fpath"
            continue
        end
        push!(dfs, df)
        push!(present_chains, c)
    end

    if length(dfs) < 2
        @warn "Dataset $sim (k=$best_k) has <2 usable chains; skipping."
        return nothing
    end

    common_params = reduce(intersect, map(names, dfs))
    if isempty(common_params)
        @warn "Dataset $sim (k=$best_k): no common parameters across chains; skipping."
        return nothing
    end
    n_keep = minimum(nrow.(dfs))
    dfs = [df[1:n_keep, common_params] for df in dfs]

    gr_dict = Dict{String,Float64}()
    for p in String.(common_params)
        mat = Array{Float64}(undef, length(dfs), n_keep)
        for (i, df) in enumerate(dfs)
            mat[i, :] = Float64.(df[!, p])
        end
        gr_dict[p] = rhat_gelman_rubin(mat)
    end
    return gr_dict
end

all_sims_gr = Vector{Dict{String,Float64}}()
sim_order   = Int[]                 
k_for_sim   = Int[]              

for row in eachrow(waic_df)
    sim    = Int(round(row.Dataset))
    best_k = parse_best_k(row.best_k)
    if best_k === nothing
        @warn "Dataset $sim has invalid best_k=$(row.best_k); skipping."
        continue
    end
    @info(@sprintf("Computing R̂ for dataset %d with best_k=%d", sim, best_k))
    gr = rhat_for_dataset(sim, best_k, chain_ids; burnin=burnin, thin=thin)
    if gr === nothing
        continue
    end
    push!(all_sims_gr, gr)
    push!(sim_order, sim)
    push!(k_for_sim, best_k)
end
write_gelman_rubin_csv(outfile, all_sims_gr)
@info "R̂ summary written to $(outfile)"
labeled_outfile = joinpath(data_dir, "rhat_summary_labeled.csv")
if !isempty(all_sims_gr)
    open(labeled_outfile, "w") do io
        println(io, "Dataset,best_k,Parameter,Rhat")
        for (idx, gr) in enumerate(all_sims_gr)
            sim = sim_order[idx]
            bk  = k_for_sim[idx]
            for (param, rhat) in gr
                @printf(io, "%d,%d,%s,%.6f\n", sim, bk, param, rhat)
            end
        end
    end
    @info "Labeled R̂ summary written to $(labeled_outfile)"
end


[ Info: Computing R̂ for dataset 1 with best_k=14
[ Info: Computing R̂ for dataset 2 with best_k=14
[ Info: Computing R̂ for dataset 3 with best_k=14
[ Info: Computing R̂ for dataset 4 with best_k=14
[ Info: Computing R̂ for dataset 5 with best_k=14
[ Info: Computing R̂ for dataset 6 with best_k=13
[ Info: Computing R̂ for dataset 7 with best_k=14
[ Info: Computing R̂ for dataset 8 with best_k=13
[ Info: Computing R̂ for dataset 9 with best_k=14
[ Info: Computing R̂ for dataset 10 with best_k=13
[ Info: Computing R̂ for dataset 11 with best_k=14
[ Info: Computing R̂ for dataset 12 with best_k=14
[ Info: Computing R̂ for dataset 13 with best_k=14
[ Info: Computing R̂ for dataset 14 with best_k=14
[ Info: Computing R̂ for dataset 15 with best_k=14
[ Info: Computing R̂ for dataset 16 with best_k=14
[ Info: Computing R̂ for dataset 17 with best_k=14
[ Info: Computing R̂ for dataset 18 with best_k=14
[ Info: Computing R̂ for dataset 19 with best_k=14
[ Info: Computing R̂ for dataset 20 with

In [ ]:
using DelimitedFiles
using Statistics
using StatsBase
using Random
using Printf

const OUTPUT_DIR        = "output"
const N_CHAINS          = 3
const BURN_IN_SAMPLES   = 500_000
const THIN_SAMPLES      = 10

param_header = ["beta","alpha","gamma"]
const N_DRAWS = 1000

# ---------------- I/O ----------------
function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function read_bestk_map(path::String)::Dict{Int,Int}
    isfile(path) || error("Cannot find $(path).")
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    H = hdr === nothing ? String[] : String.(vec(hdr))
    cols = Dict(name => findfirst(==(name), H) for name in H)
    haskey(cols, "Dataset") || error("Column 'Dataset' not found in $(path)")
    haskey(cols, "best_k")  || error("Column 'best_k' not found in $(path)")

    bestk = Dict{Int,Int}()
    for r in 1:size(data,1)
        ds_val = data[r, cols["Dataset"]]
        bk_val = data[r, cols["best_k"]]
        ds = ds_val isa Number ? Int(round(ds_val)) : parse(Int, String(ds_val))
        if (bk_val isa Number && !isnan(bk_val))
            bk = Int(round(bk_val))
        else
            try
                bk = parse(Int, String(bk_val))
            catch
                @warn "Row $r has invalid best_k=$(bk_val); skip dataset=$ds"
                continue
            end
        end
        bestk[ds] = bk
    end
    return bestk
end

function load_samples_for_sim_k(sim_id::Int, chain::Int, k::Int, outdir::String)
    path = joinpath(outdir, "samples_sim_$(sim_id)_chain_$(chain)_k$(k).csv")
    if !isfile(path)
        @warn "Missing samples file: $path"
        return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if size(M,1) <= BURN_IN_SAMPLES
        @warn "File $path has only $(size(M,1)) rows (<= burn-in)."
        return Array{Float64}(undef, 0, 0)
    end
    idxs = collect(BURN_IN_SAMPLES+1:THIN_SAMPLES:size(M,1))
    return M[idxs, :]
end

function summarize_posterior(samples::AbstractMatrix{<:Real}, header_cont::Vector{String})
    summary_dict = Dict{String, Any}()
    for (i, p_name) in enumerate(header_cont)
        param_samples = samples[:, i]
        if p_name == "k_max"
            k = mode(round.(Int, param_samples))
            freq = count(==(k), round.(Int, param_samples)) / length(param_samples)
            summary_dict[p_name] = Dict("mode" => k, "frequency" => freq)
        else
            med = median(param_samples)
            q = quantile(param_samples, [0.025, 0.975])
            summary_dict[p_name] = Dict("median" => med, "95%CI_low" => q[1], "95%CI_high" => q[2])
        end
    end
    return summary_dict
end

function write_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        DelimitedFiles.writedlm(io, M, ',')
    end
end

function write_summary_csv(path::String, summary_data::Vector{Dict{String,Any}})
    open(path, "w") do io
        println(io, "Dataset,Parameter,Median_or_Mode,CI_Low_or_Frequency,CI_High")
        for (dataset_idx, dict) in enumerate(summary_data)
            for (param, vals) in dict
                if haskey(vals, "mode")
                    println(io, "$dataset_idx,$param,$(vals["mode"]),$(vals["frequency"]),")
                else
                    println(io, "$dataset_idx,$param,$(vals["median"]),$(vals["95%CI_low"]),$(vals["95%CI_high"])")
                end
            end
        end
    end
end


Random.seed!(2025)

bestk_csv = joinpath(OUTPUT_DIR, "waic_best3_mergedchains.csv")
bestk_map = read_bestk_map(bestk_csv)
isempty(bestk_map) && error("No (Dataset -> best_k) pairs found in $(bestk_csv)")

posterior_summaries = Dict{String,Any}[]

for sim in sort(collect(keys(bestk_map)))
    k = bestk_map[sim]
    @info "Processing dataset $sim with best_k=$k ..."

    mats = Matrix{Float64}[]
    for c in 1:N_CHAINS
        M = load_samples_for_sim_k(sim, c, k, OUTPUT_DIR)
        if size(M,1) > 0
            push!(mats, M)
        else
            @warn "No usable samples for sim=$sim, chain=$c, k=$k"
        end
    end
    samples_bt = isempty(mats) ? Array{Float64}(undef,0,0) : vcat(mats...)

    if size(samples_bt,1) == 0
        @warn "Skip dataset $sim (best_k=$k) due to empty samples after burn/thin."
        push!(posterior_summaries, Dict{String,Any}())
        continue
    end

    # Posterior summary
    push!(posterior_summaries, summarize_posterior(samples_bt, param_header))

    # Posterior draws
    n_rows = size(samples_bt, 1)
    replace_flag = n_rows < N_DRAWS
    draw_indices = sample(1:n_rows, N_DRAWS; replace=replace_flag)
    draw_params  = samples_bt[draw_indices, :]

    draws_outfile = joinpath(OUTPUT_DIR, @sprintf("posterior_draws_sim_%d_k%d.csv", sim, k))
    hdr = vcat(["row_index"], param_header)
    Mout = hcat(Float64.(draw_indices), draw_params)
    write_csv(draws_outfile, hdr, Mout)
end

write_summary_csv(joinpath(OUTPUT_DIR, "posterior_summary_by_dataset_bestk.csv"), posterior_summaries)

@info "Done. Outputs written in $(OUTPUT_DIR): posterior_draws_sim_*_k*.csv, posterior_summary_by_dataset_bestk.csv"


[ Info: Processing dataset 1 with best_k=14 ...
[ Info: Processing dataset 2 with best_k=14 ...
[ Info: Processing dataset 3 with best_k=14 ...
[ Info: Processing dataset 4 with best_k=14 ...
[ Info: Processing dataset 5 with best_k=14 ...
[ Info: Processing dataset 6 with best_k=13 ...
[ Info: Processing dataset 7 with best_k=14 ...
[ Info: Processing dataset 8 with best_k=13 ...
[ Info: Processing dataset 9 with best_k=14 ...
[ Info: Processing dataset 10 with best_k=13 ...
[ Info: Processing dataset 11 with best_k=14 ...
[ Info: Processing dataset 12 with best_k=14 ...
[ Info: Processing dataset 13 with best_k=14 ...
[ Info: Processing dataset 14 with best_k=14 ...
[ Info: Processing dataset 15 with best_k=14 ...
[ Info: Processing dataset 16 with best_k=14 ...
[ Info: Processing dataset 17 with best_k=14 ...
[ Info: Processing dataset 18 with best_k=14 ...
[ Info: Processing dataset 19 with best_k=14 ...
[ Info: Processing dataset 20 with best_k=14 ...
[ Info: Done. Outputs written

In [ ]:
using Random, Distributions, Statistics, Printf, DelimitedFiles
using Plots
Random.seed!(2025)
# ---------- helpers ----------
function clamp01(x)
    x < 0 ? 0.0 : (x > 1 ? 1.0 : x)
end

# a_t = 1 - (1 - ψ_t)^(1/α)
function baseline_alarm(psi, alpha)
    psi = clamp01(psi)
    alpha <= 0 && error("alpha must be > 0")
    1 - (1 - psi)^(1/alpha)
end

psi_memoryless(Istar_hist, N) = isempty(Istar_hist) ? 0.0 : Istar_hist[end] / N

function psi_sliding(Istar_hist, N, k_max)
    L = length(Istar_hist)
    L == 0 && return 0.0
    m = min(k_max, L)
    mean(@view(Istar_hist[(L - m + 1):L])) / N
end

function psi_powerlaw(Istar_hist, N, lambda_P)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 1:L
        w = j^(-lambda_P)
        num  += w * Istar_hist[end - j + 1]
    end
    num / N
end

function psi_exponential(Istar_hist, N, lambda_E)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 0:(L - 1)
        w = exp(-lambda_E * j)
        num  += w * Istar_hist[end - j]
    end
    num / N
end

function psi_reciprocal(Istar_hist, N, lambda_R)
    lambda_R <= 0 && error("lambda_R must be > 0")
    L = length(Istar_hist)
    L == 0 && return 0.0
    num = 0.0
    @inbounds for j in 0:(L - 1)
        w = 1.0 / (1.0 + lambda_R * j)
        num += w * Istar_hist[end - j]
    end
    num / N
end

# ---------- simulate one epidemic ----------
function simulate_epidemic(N, I0, tau, beta, alpha, rateI;
                           mechanism::Symbol = :sliding,
                           k_max::Union{Nothing,Int}=nothing,
                           lambda_P::Union{Nothing,Float64}=nothing,
                           lambda_E::Union{Nothing,Float64}=nothing,
                           lambda_R::Union{Nothing,Float64}=nothing)

    S      = zeros(Int, tau + 1)
    I      = zeros(Int, tau + 1)
    Istar  = zeros(Int, tau)
    Rstar  = zeros(Int, tau)
    psi    = zeros(Float64, tau)
    alarm  = zeros(Float64, tau)
    probSI = zeros(Float64, tau)
    probIR = 1 - exp(-rateI)

    S[1] = N - I0
    I[1] = I0

    # t = 1
    psi[1]   = 0.0
    alarm[1] = baseline_alarm(psi[1], alpha)
    pSI      = clamp01(1 - exp(-beta * (1 - alarm[1]) * (I[1] / N)))
    probSI[1] = pSI
    Istar[1] = rand(Binomial(S[1], pSI))
    Rstar[1] = rand(Binomial(I[1], probIR))
    S[2]     = S[1] - Istar[1]
    I[2]     = I[1] + Istar[1] - Rstar[1]

    # t = 2..tau
    for t in 2:tau
        hist = @view Istar[1:(t-1)]
        ψt = if mechanism === :memoryless
            psi_memoryless(hist, N)
        elseif mechanism === :sliding
            isnothing(k_max) && error("k_max must be provided for :sliding")
            psi_sliding(hist, N, k_max)
        elseif mechanism === :powerlaw
            isnothing(lambda_P) && error("lambda_P must be provided for :powerlaw")
            psi_powerlaw(hist, N, lambda_P)
        elseif mechanism === :exponential
            isnothing(lambda_E) && error("lambda_E must be provided for :exponential")
            psi_exponential(hist, N, lambda_E)
        elseif mechanism === :reciprocal
            isnothing(lambda_R) && error("lambda_R must be provided for :reciprocal")
            psi_reciprocal(hist, N, lambda_R)
        else
            error("Unknown mechanism: $mechanism")
        end
        psi[t]   = ψt
        alarm[t] = baseline_alarm(ψt, alpha)
        pSI      = clamp01(1 - exp(-beta * (1 - alarm[t]) * (I[t] / N)))
        probSI[t] = pSI

        Istar[t] = rand(Binomial(S[t], pSI))
        Rstar[t] = rand(Binomial(I[t], probIR))

        S[t+1] = S[t] - Istar[t]
        I[t+1] = I[t] + Istar[t] - Rstar[t]
    end

    return Dict(
        :Istar => Istar,
        :Rstar => Rstar,
        :S     => S,
        :I     => I,
        :psi   => psi,
        :alarm => alarm,
        :probSI => probSI,
        :probIR => probIR
    )
end

# ---------- small I/O helpers ----------
function write_matrix_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        for i in 1:size(M,1)
            println(io, join(M[i, :], ","))
        end
    end
end

OUTPUT_DIR = "output"
DATA_DIR   = "data"
TAU = 50
N_pop, I0 = 1_000_000, 10
MECHANISM = :sliding
params = ["beta","alpha","gamma"]   

function read_draws(path)
    data,hdr = readdlm(path,',',header=true)
    Matrix{Float64}(data),vec(hdr)
end

function idxmap(header,names)
    H=Dict(n=>i for (i,n) in enumerate(header))
    Dict(n=>get(H,n,nothing) for n in names)
end

function summarize(M)
    τ=size(M,2); med=Float64[]; lo=Float64[]; hi=Float64[] 
    for t=1:τ
        col=M[:,t]; push!(med,median(col))
        q=quantile(col,[0.025,0.975]); push!(lo,q[1]); push!(hi,q[2])
    end
    med,lo,hi
end

function write_csv(path,hdr,rows)
    open(path,"w") do io
        println(io,join(hdr,",")); foreach(r->println(io,join(r,",")),rows)
    end
end

function avg_series(list)
    τ=length(list[1]); [mean([s[t] for s in list]) for t=1:τ]
end

# ---------- read best_k per dataset ----------
function read_bestk_map(path::String)::Dict{Int,Int}
    isfile(path) || error("Cannot find $(path)")
    data, hdr = readdlm(path, ',', header=true)
    H = Dict(n=>i for (i,n) in enumerate(vec(hdr)))
    haskey(H,"Dataset") || error("best3 file missing 'Dataset'")
    haskey(H,"best_k")  || error("best3 file missing 'best_k'")
    bestk = Dict{Int,Int}()
    for r in 1:size(data,1)
        ds = Int(round(data[r, H["Dataset"]]))
        bk = data[r, H["best_k"]]
        if bk isa Number && !isnan(bk)
            bestk[ds] = Int(round(bk))
        else
            @warn "Dataset $ds has invalid best_k=$bk; skipping"
        end
    end
    return bestk
end

best3_path = joinpath(OUTPUT_DIR, "waic_best3_mergedchains.csv")
bestk_map  = read_bestk_map(best3_path)

sim_ids = sort(collect(keys(bestk_map)))

istarM=[]; istarL=[]; istarH=[]
alarmM=[]; alarmL=[]; alarmH=[]

for sim in sim_ids
    kbest = bestk_map[sim]
    @info "Processing dataset $sim with best_k=$kbest ..."
    draws_path = "$(OUTPUT_DIR)/posterior_draws_sim_$(sim)_k$(kbest).csv"
    isfile(draws_path) || (@warn "Missing draws: $draws_path"; continue)

    M,hdr = read_draws(draws_path)
    col   = idxmap(hdr, params)
    n     = size(M,1)
    istar = zeros(n, TAU)
    alarm = zeros(n, TAU)

    for k in 1:n
        β = M[k, col["beta"]]
        α = M[k, col["alpha"]]
        γ = M[k, col["gamma"]]   

        simres = simulate_epidemic(N_pop, I0, TAU, β, α, γ;
                                   mechanism = MECHANISM,
                                   k_max = kbest)

        istar[k,:] = simres[:Istar]
        alarm[k,:] = simres[:alarm]
    end

    m,l,h = summarize(istar)
    write_csv("$(OUTPUT_DIR)/Istar_stats_sim_$(sim)_k$(kbest).csv",
              ["Day","median","ci_low","ci_high"],
              [[t,m[t],l[t],h[t]] for t=1:TAU])

    m,l,h = summarize(alarm)
    write_csv("$(OUTPUT_DIR)/alarm_stats_sim_$(sim)_k$(kbest).csv",
              ["Day","median","ci_low","ci_high"],
              [[t,m[t],l[t],h[t]] for t=1:TAU])

    push!(istarM,m); push!(istarL,l); push!(istarH,h)
    push!(alarmM,m); push!(alarmL,l); push!(alarmH,h)
end

function average_files(pattern)
    files = sort(filter(f -> occursin(pattern, f), readdir(OUTPUT_DIR; join=true)))
    isempty(files) && error("No files match pattern '$pattern' under $(OUTPUT_DIR)")
    matrices = [DelimitedFiles.readdlm(f, ',', header=true)[1] for f in files]
    reduce(+, matrices) ./ length(matrices)
end

function add_truth_and_save(meanmat, truthfile, truthcolname, outfile)
    data, hdr = readdlm(joinpath(DATA_DIR, truthfile), ',', header=true)
    H = Dict(n=>i for (i,n) in enumerate(vec(hdr)))
    haskey(H,truthcolname) || error("Truth file $(truthfile) missing column $(truthcolname)")
    truth = data[:, H[truthcolname]]
    out = hcat(meanmat, truth)
    open(joinpath(OUTPUT_DIR, outfile), "w") do io
        println(io, "Day,median,ci_low,ci_high,truth")
        writedlm(io, out, ',')
    end
end


# Istar
istar_mean = average_files("Istar_stats_sim_")
add_truth_and_save(istar_mean, "sliding_kmax14_mean_incidence.csv", "mean_Istar", "Istar_stats_mean.csv")
# Alarm
alarm_mean = average_files("alarm_stats_sim_")
add_truth_and_save(alarm_mean, "sliding_kmax14_mean_alarm.csv", "mean_alarm", "alarm_stats_mean.csv")


[ Info: Processing dataset 1 with best_k=14 ...
[ Info: Processing dataset 2 with best_k=14 ...
[ Info: Processing dataset 3 with best_k=14 ...
[ Info: Processing dataset 4 with best_k=14 ...
[ Info: Processing dataset 5 with best_k=14 ...
[ Info: Processing dataset 6 with best_k=13 ...
[ Info: Processing dataset 7 with best_k=14 ...
[ Info: Processing dataset 8 with best_k=13 ...
[ Info: Processing dataset 9 with best_k=14 ...
[ Info: Processing dataset 10 with best_k=13 ...
[ Info: Processing dataset 11 with best_k=14 ...
[ Info: Processing dataset 12 with best_k=14 ...
[ Info: Processing dataset 13 with best_k=14 ...
[ Info: Processing dataset 14 with best_k=14 ...
[ Info: Processing dataset 15 with best_k=14 ...
[ Info: Processing dataset 16 with best_k=14 ...
[ Info: Processing dataset 17 with best_k=14 ...
[ Info: Processing dataset 18 with best_k=14 ...
[ Info: Processing dataset 19 with best_k=14 ...
[ Info: Processing dataset 20 with best_k=14 ...
